# 🐾 Assistente Veterinário — Pet Family

Protótipo do assistente de IA para o app **Pet Family** (Challenge CLYVO VET — FIAP ADS 2TDSPG 2026).

O assistente responde dúvidas sobre saúde, vacinas, alimentação e bem-estar do pet do tutor, usando os dados do animal para personalizar as respostas.

**Equipe:** Felipe Kirschner Modesto (RM561810) · João Victor Luiz Oliveira Resende (RM565139) · Pedro Vaz Ferreira (RM566551) · Vitor Dias dos Santos (RM565422)

---

## Estrutura do notebook

1. Configuração (instalar SDK e autenticar)
2. Dados do pet (simulando o que viria do app)
3. System prompt personalizado
4. Conversa simples (sem histórico)
5. Conversa com histórico (Lab 2)
6. Resposta estruturada em JSON (Lab 3)

## 1. Configuração

Instala o SDK do Gemini e configura a chave de API.

> Obtenha sua chave gratuita em: https://aistudio.google.com/apikey

In [ ]:
# instala o SDK do Google Gemini
!pip install -q google-genai

In [ ]:
from google import genai
from google.colab import userdata

# carrega a chave salva nos Secrets do Colab (ícone de chave na barra lateral)
# nome do secret: GEMINI_API_KEY
chave_api = userdata.get('GEMINI_API_KEY')
cliente = genai.Client(api_key=chave_api)

print('SDK configurado com sucesso!')

## 2. Dados do pet

No app real, esses dados vêm do AsyncStorage do React Native (perfil do pet cadastrado pelo tutor).

Aqui simulamos um pet de exemplo para testar o assistente.

In [ ]:
# dados do pet — no app real, viriam do AsyncStorage
dados_pet = {
    "nome": "Bolinha",
    "especie": "cachorro",
    "raca": "Golden Retriever",
    "idade": "3 anos",
    "peso": "28 kg",
    "sexo": "macho",
    "notas": "alérgico a frango, última vacina em março de 2026"
}

print(f"Pet carregado: {dados_pet['nome']} ({dados_pet['raca']}, {dados_pet['idade']})")

## 3. System prompt personalizado

O system prompt define o comportamento do assistente e injeta os dados do pet para personalizar as respostas.

Essa é a técnica do **Lab 1** (engenharia de prompts): dar contexto, persona e restrições ao modelo.

In [ ]:
def montar_system_prompt(pet):
    """Monta o system prompt com os dados do pet."""
    return f"""
Você é o assistente veterinário virtual do app Pet Family, chamado PetDoc.

Seu papel é ajudar tutores de pets com dúvidas sobre saúde, vacinas, alimentação,
comportamento e bem-estar dos seus animais. Responda de forma clara, acolhedora
e acessível — o tutor não é veterinário.

Dados do pet do tutor:
- Nome: {pet['nome']}
- Espécie: {pet['especie']}
- Raça: {pet['raca']}
- Idade: {pet['idade']}
- Peso: {pet['peso']}
- Sexo: {pet['sexo']}
- Observações: {pet['notas']}

Regras importantes:
1. Sempre considere os dados do pet acima ao responder.
2. Para situações de emergência ou sintomas graves, oriente o tutor a procurar
   um veterinário imediatamente. Nunca substitua uma consulta presencial.
3. Não invente informações médicas. Se não souber, diga que não sabe.
4. Seja direto e objetivo. Evite respostas muito longas.
5. Chame o pet sempre pelo nome ({pet['nome']}).
"""

system_prompt = montar_system_prompt(dados_pet)
print(system_prompt)

## 4. Conversa simples (sem histórico)

Primeiro teste: uma pergunta direta, sem manter contexto entre mensagens.

In [ ]:
from google.genai import types

MODELO = "gemini-3.8-flash"

def perguntar(pergunta):
    """Faz uma pergunta simples ao assistente (sem histórico)."""
    resposta = cliente.models.generate_content(
        model=MODELO,
        contents=pergunta,
        config=types.GenerateContentConfig(
            system_instruction=system_prompt
        )
    )
    return resposta.text

# teste 1 — pergunta geral
print("Pergunta: Quais vacinas o Bolinha precisa tomar?")
print("-" * 50)
print(perguntar("Quais vacinas o Bolinha precisa tomar?"))

In [ ]:
# teste 2 — sintoma preocupante
print("Pergunta: O Bolinha está vomitando muito desde ontem, o que pode ser?")
print("-" * 50)
print(perguntar("O Bolinha está vomitando muito desde ontem, o que pode ser?"))

In [ ]:
# teste 3 — considera a alergia do pet
print("Pergunta: Posso dar ração de frango para o Bolinha?")
print("-" * 50)
print(perguntar("Posso dar ração de frango para o Bolinha?"))

## 5. Conversa com histórico (Lab 2)

Aqui implementamos o histórico de conversa: o assistente lembra do que foi dito anteriormente na mesma sessão.

No app real, esse histórico seria mantido no estado do React Native (últimas N mensagens).

In [ ]:
# inicia uma sessão de chat com histórico
chat = cliente.chats.create(
    model=MODELO,
    config=types.GenerateContentConfig(
        system_instruction=system_prompt
    )
)

def conversar(mensagem):
    """Envia uma mensagem mantendo o histórico da conversa."""
    resposta = chat.send_message(mensagem)
    return resposta.text

def mostrar_historico():
    """Exibe o histórico da conversa atual."""
    historico = chat.get_history()
    print(f"\nHistórico ({len(historico)} mensagens):")
    for msg in historico:
        papel = "Tutor" if msg.role == "user" else "PetDoc"
        texto = msg.parts[0].text[:80] + "..." if len(msg.parts[0].text) > 80 else msg.parts[0].text
        print(f"  [{papel}]: {texto}")

In [ ]:
# turno 1
print("Tutor: O Bolinha está coçando muito a orelha.")
print("-" * 50)
print(conversar("O Bolinha está coçando muito a orelha."))

In [ ]:
# turno 2 — pergunta de follow-up sem repetir o contexto
print("Tutor: E isso pode ser por causa da alergia que ele tem?")
print("-" * 50)
print(conversar("E isso pode ser por causa da alergia que ele tem?"))

In [ ]:
# turno 3
print("Tutor: Preciso ir ao veterinário ou posso esperar mais um pouco?")
print("-" * 50)
print(conversar("Preciso ir ao veterinário ou posso esperar mais um pouco?"))

# exibe o histórico completo da sessão
mostrar_historico()

## 6. Resposta estruturada em JSON (Lab 3)

Agora pedimos ao modelo para retornar um JSON estruturado além da resposta textual.

Isso permite que o app exiba alertas visuais de urgência e tome decisões com base nos campos retornados.

Campos retornados:
- `resposta` — texto para exibir no chat
- `urgencia` — `"baixa"`, `"media"` ou `"alta"`
- `recomenda_vet` — `true` se deve procurar veterinário

In [ ]:
import json

# system prompt estendido para retornar JSON
system_prompt_json = montar_system_prompt(dados_pet) + """

FORMATO DE RESPOSTA OBRIGATÓRIO:
Responda SEMPRE com um JSON válido, sem nenhum texto fora do JSON. Use este formato:
{
  "resposta": "texto da resposta para o tutor",
  "urgencia": "baixa" | "media" | "alta",
  "recomenda_vet": true | false
}

Critérios de urgência:
- "baixa": dúvidas gerais, cuidados preventivos, comportamento normal
- "media": sintomas leves que merecem atenção nos próximos dias
- "alta": sintomas graves, emergências, risco de vida — oriente ir ao vet imediatamente
"""

def perguntar_estruturado(pergunta):
    """Retorna a resposta do assistente como dicionário Python."""
    resposta = cliente.models.generate_content(
        model=MODELO,
        contents=pergunta,
        config=types.GenerateContentConfig(
            system_instruction=system_prompt_json,
            response_mime_type="application/json"
        )
    )
    dados = json.loads(resposta.text)
    return dados

def exibir_resposta(dados):
    """Simula como o app exibiria a resposta."""
    cores_urgencia = {"baixa": "🟢", "media": "🟡", "alta": "🔴"}
    icone = cores_urgencia.get(dados["urgencia"], "⚪")

    print(f"PetDoc: {dados['resposta']}")
    print(f"Urgência: {icone} {dados['urgencia'].upper()}")
    if dados["recomenda_vet"]:
        print("⚕️  Recomenda visita ao veterinário")
    print("\nJSON retornado:", json.dumps(dados, ensure_ascii=False, indent=2))

In [ ]:
# teste 1 — urgência baixa
print("Pergunta: Com qual frequência devo dar banho no Bolinha?")
print("-" * 50)
exibir_resposta(perguntar_estruturado("Com qual frequência devo dar banho no Bolinha?"))

In [ ]:
# teste 2 — urgência média
print("Pergunta: O Bolinha não quer comer há dois dias, isso é normal?")
print("-" * 50)
exibir_resposta(perguntar_estruturado("O Bolinha não quer comer há dois dias, isso é normal?"))

In [ ]:
# teste 3 — urgência alta
print("Pergunta: O Bolinha convulsionou agora há pouco, o que faço?")
print("-" * 50)
exibir_resposta(perguntar_estruturado("O Bolinha convulsionou agora há pouco, o que faço?"))

## ✅ Resumo

| Seção | Técnica aplicada | Lab de referência |
|-------|-----------------|-------------------|
| System prompt personalizado | Engenharia de prompts com dados dinâmicos | Lab 1 |
| Conversa simples | Chamada direta à API Gemini | Lab 1 |
| Histórico de conversa | `chats.create()` com histórico mantido | Lab 2 |
| Resposta estruturada | `response_mime_type: application/json` + JSON parsing | Lab 3 |

### Próximos passos

- Transformar este protótipo em uma **API FastAPI** (Lab 3.5)
- Integrar a API ao app **PET-FAMILY-MOBILE** (React Native)
- Commitar o notebook e a API no repositório **PET-FAMILY-IA**